In [2]:
import os, sys
from pathlib import Path
# Allow overriding base dir via environment variable BASE_DIR
env_base = os.environ.get('BASE_DIR')
if env_base:
    base_dir = Path(env_base).resolve()
else:
    # Search upward from current working directory for a folder that contains 'module'
    p = Path.cwd().resolve()
    found = None
    for _ in range(20):
        if (p / 'module').is_dir():
            found = p
            break
        if p.parent == p:
            break
        p = p.parent
    base_dir = found or Path.cwd().resolve()
# Ensure base_dir is on sys.path so 'module' can be imported
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))
print(f"Using base_dir: {base_dir}")
import pandas as pd
df = pd.read_csv(str(base_dir / 'data' / 'data.csv'))
df.head()

Using base_dir: /home/nguyenhuynh/Documents/appcuuho/be


,stt,content,reactions,replies,posted_time,location,contact,disaster_type,affected_people,infrastructure_damage,needs,is_isolated,deathly,has_emergency_terms,has_emotional_terms,has_emphasis,label,confident
0,1,"CẦN HỖ TRỢ LƯƠNG THỰC THỰC PHẨM, SẠC, ĐÈN PIN ...",82,29,1w,"nhà số 9 kiệt 34 An Ninh, phường Kim Long, thà...",['0339 186 860'],lũ lụt,4.0,False,"['lương thực & nước uống', 'liên lạc & thông t...",True,False,False,False,True,CAO,CAO
1,2,"CẦN CỨU KhẨN CẤP Khu vực dân Vĩnh Điện, Điện B...",0,0,1w,"Vĩnh Điện, Điện Bàn, Quảng Nam",[],lũ lụt,NaN,True,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",True,True,True,True,False,CAO,CAO
2,4,"-Người thân em đang ở địa chỉ định vị này,sáng...",0,0,1w,"thôn Tây An, xã Gò Nổi, Đà Nẵng (Quảng Nam cũ)",['098 2303418'],lũ lụt,NaN,True,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",True,True,False,False,True,CAO,CAO
3,5,"Anh chồng mình là Nguyễn Xuân Năm, bị ngộ độc ...",0,0,1w,quán dê Phố Núi,['https://www.facebook.com/share/p/1H6dRCGLDw/...,lũ lụt,1.0,False,['phương tiện & cứu hộ'],True,True,False,False,True,CAO,THẤP
4,6,"Đào Thị Quy. Nhà có 1 bé 4 tuổi, 1 bé 6 tuổi, ...",1,0,17h,"đội 9, thôn Hòa Nghĩa, Xã Mỹ Hiệp, huyện Phù M...",['0374036746'],lũ lụt,6.0,True,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",True,True,True,False,False,CAO,CAO


In [6]:
%pip install langchain langchain-community langchain-openai

/home/nguyenhuynh/Documents/appcuuho/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import re

def clean_data(input_file="data.csv", output_file="data_cleaned.csv"):
    # 1. Đọc dữ liệu
    df = pd.read_csv(input_file)

    # 2. Ghi nhận số lượng dòng trước khi lọc
    count_before = len(df)

    # 3. Lọc trùng dựa trên cột 'content'
    df_cleaned = df.drop_duplicates(subset=['content'], keep='first').copy()

    # 4. Hàm kiểm tra và mã hóa cột 'contact'
    def contact_has_phone(val):
        if pd.isna(val) or val == '[]':
            return 0
        s = str(val)
        digits = re.sub(r'[^0-9]', '', s)
        return 1 if len(digits) >= 7 else 0

    # Áp dụng mã hóa cho cột 'contact'
    if 'contact' in df_cleaned.columns:
        df_cleaned['contact'] = df_cleaned['contact'].apply(contact_has_phone)
    else:
        df_cleaned['contact'] = 0

    # 5. Ghi nhận số lượng dòng sau khi lọc
    count_after = len(df_cleaned)

    # 6. Lưu kết quả ra file mới
    df_cleaned.to_csv(output_file, index=False, encoding='utf-8-sig')

    # 7. In kết quả kiểm tra
    print(f"Số dòng ban đầu: {count_before}")
    print(f"Số dòng sau khi lọc trùng: {count_after}")
    print(f"Đã xóa: {count_before - count_after} dòng trùng lặp.")
    print("\n--- 5 dòng đầu tiên của dữ liệu đã sạch (contact đã được mã hóa) ---")
    print(df_cleaned.head())

    return df_cleaned

input_file = "/home/nguyenhuynh/Documents/appcuuho/be/data/dataset/data.csv"

clean_data(input_file, "data_cleaned.csv")



Số dòng ban đầu: 841
Số dòng sau khi lọc trùng: 483
Đã xóa: 358 dòng trùng lặp.

--- 5 dòng đầu tiên của dữ liệu đã sạch (contact đã được mã hóa) ---
   stt                                            content  reactions  replies  \
0    1  CẦN HỖ TRỢ LƯƠNG THỰC THỰC PHẨM, SẠC, ĐÈN PIN ...         82       29   
1    2  CẦN CỨU KhẨN CẤP Khu vực dân Vĩnh Điện, Điện B...          0        0   
2    4  -Người thân em đang ở địa chỉ định vị này,sáng...          0        0   
3    5  Anh chồng mình là Nguyễn Xuân Năm, bị ngộ độc ...          0        0   
4    6  Đào Thị Quy. Nhà có 1 bé 4 tuổi, 1 bé 6 tuổi, ...          1        0   

  posted_time                                           location  contact  \
0          1w  nhà số 9 kiệt 34 An Ninh, phường Kim Long, thà...        1   
1          1w                     Vĩnh Điện, Điện Bàn, Quảng Nam        0   
2          1w     thôn Tây An, xã Gò Nổi, Đà Nẵng (Quảng Nam cũ)        1   
3          1w                                    quán d

,stt,content,reactions,replies,posted_time,location,contact,disaster_type,affected_people,infrastructure_damage,needs,is_isolated,deathly,has_emergency_terms,has_emotional_terms,has_emphasis,label,confident
0,1,"CẦN HỖ TRỢ LƯƠNG THỰC THỰC PHẨM, SẠC, ĐÈN PIN ...",82,29,1w,"nhà số 9 kiệt 34 An Ninh, phường Kim Long, thà...",1,lũ lụt,4.0,False,"['lương thực & nước uống', 'liên lạc & thông t...",True,False,False,False,True,CAO,CAO
1,2,"CẦN CỨU KhẨN CẤP Khu vực dân Vĩnh Điện, Điện B...",0,0,1w,"Vĩnh Điện, Điện Bàn, Quảng Nam",0,lũ lụt,NaN,True,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",True,True,True,True,False,CAO,CAO
2,4,"-Người thân em đang ở địa chỉ định vị này,sáng...",0,0,1w,"thôn Tây An, xã Gò Nổi, Đà Nẵng (Quảng Nam cũ)",1,lũ lụt,NaN,True,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",True,True,False,False,True,CAO,CAO
3,5,"Anh chồng mình là Nguyễn Xuân Năm, bị ngộ độc ...",0,0,1w,quán dê Phố Núi,0,lũ lụt,1.0,False,['phương tiện & cứu hộ'],True,True,False,False,True,CAO,THẤP
4,6,"Đào Thị Quy. Nhà có 1 bé 4 tuổi, 1 bé 6 tuổi, ...",1,0,17h,"đội 9, thôn Hòa Nghĩa, Xã Mỹ Hiệp, huyện Phù M...",1,lũ lụt,6.0,True,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",True,True,True,False,False,CAO,CAO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
820,1090,- Tên: Võ Ngọc Tuấn - Tình trạng ngập toàn bộ ...,0,0,13h,"xóm 21, thôn Vinh Quang 2, xã Phước Sơn, huyện...",0,lũ lụt,NaN,False,"['lương thực & nước uống', 'thuốc men & hỗ trợ...",False,True,False,False,False,TRUNG BÌNH,THẤP
822,1093,"A oi, nhờ a vào hỗ trợ cho gd dượng e, nhà 4đứ...",4,0,5d,Thôn 3 Cẩm Thạch Cũ Cẩm Xuyên,1,lũ lụt,4.0,True,"['lương thực & nước uống', 'chỗ ở & nơi trú ẩn...",False,False,False,False,False,CAO,CAO
823,1094,"Bame mình đi làm ăn xa , 2 đứa em mình bị mắc ...",0,0,1w,12b kiệt 110 Nhật Lệ,1,lũ lụt,2.0,True,"['lương thực & nước uống', 'liên lạc & thông t...",True,False,True,False,False,TRUNG BÌNH,CAO
824,1095,KHẨN CẤP: LẠI LÀM PHIỀN MỌI NGƯỜI NHỜ HỖ TRỢ G...,0,0,1w,"cây đa Chợ Đình, Xuyên Đông, gần nhà thờ Nguyễ...",1,lũ lụt,2.0,False,"['lương thực & nước uống', 'phương tiện & cứu ...",True,True,True,False,True,CAO,CAO


In [4]:
import pandas as pd
import json

from pydantic import BaseModel
from module.agent import Agent

agent = Agent()

# --- PROMPT VÀ XỬ LÝ (SYNCHRONOUS) ---
prompt_template = """
Bạn là một trợ lý phân tích dữ liệu cứu hộ. Hãy bóc tách địa chỉ sau thành định dạng JSON.
Địa chỉ: {location}

Yêu cầu trả về JSON với các trường:
- "tinh_tp": Tỉnh hoặc Thành phố trực thuộc trung ương.
- "quan_huyen": Quận, Huyện hoặc Thị xã.
- "phuong_xa": Phường, Xã hoặc Thị trấn.
- "nha": Số nhà, ngõ, tên đường hoặc các mô tả chi tiết vị trí nhỏ hơn xã.

Chỉ trả về duy nhất khối JSON, không giải thích gì thêm.
Kết quả JSON:
"""
class Response(BaseModel):
    tinh_tp: str
    quan_huyen: str
    phuong_xa: str
    nha: str

def parse_address_local(location: str) -> dict:
    if pd.isna(location) or str(location).strip() == "":
        return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}
    try:
        user_prompt = prompt_template.format(location=location)
        # synchronous call to the agent
        message = agent.generate_sync(user_prompt=user_prompt, response_schema=Response)
        print(message)
        if hasattr(message, 'dict'):
            return message.dict()
        return dict(message)
    except Exception as e:
        print(f"Lỗi dòng: {location} -> {e}")
        return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}

# --- CHẠY TRÊN FILE DATA_CLEANED.CSV ---
df = pd.read_csv('data_cleaned.csv')

print(f"Bắt đầu xử lý {len(df)} dòng địa chỉ...")

# Áp dụng hàm bóc tách một cách tuần tự (synchronous)
results = [parse_address_local(loc) for loc in df['location'].tolist()]

# Chuyển list các dictionary thành các cột riêng biệt
df_address_cols = pd.json_normalize(results)

# Gộp vào dataframe gốc
df_final = pd.concat([df, df_address_cols], axis=1)

# Lưu kết quả
df_final.to_csv('data_encoded_local.csv', index=False, encoding='utf-8-sig')

print("Hoàn thành! Kiểm tra file: data_encoded_local.csv")

Bắt đầu xử lý 483 dòng địa chỉ...
tinh_tp='Thừa Thiên Huế' quan_huyen='Huế' phuong_xa='Kim Long' nha='nhà số 9 kiệt 34 An Ninh'


/tmp/ipykernel_982439/1213851596.py:38: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return message.dict()


tinh_tp='Quảng Nam' quan_huyen='Điện Bàn' phuong_xa='Vĩnh Điện' nha=''
tinh_tp='Quảng Nam' quan_huyen='Đà Nẵng' phuong_xa='Gò Nổi' nha='thôn Tây An'
tinh_tp='Tỉnh Phú Thọ' quan_huyen='Thị xã Phú Thọ' phuong_xa='Phường Núi' nha='Quán Dê'
tinh_tp='Bình Định' quan_huyen='Phù Mỹ' phuong_xa='Mỹ Hiệp' nha='đội 9, thôn Hòa Nghĩa'
tinh_tp='Thừa Thiên Huế' quan_huyen='TP Huế' phuong_xa='Kim Long' nha='55 Vạn Xuân'
tinh_tp='Quảng Bình' quan_huyen='Quảng Vinh' phuong_xa='Đan Điền' nha='Đông Lâm'
tinh_tp='Không xác định' quan_huyen='Không xác định' phuong_xa='Không xác định' nha='https://maps.app.goo.gl/y3LvkwDAMYHvfRos5'
tinh_tp='Điện Biên' quan_huyen='Nam' phuong_xa='Bắc' nha=''


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import re
import json
from pydantic import BaseModel
from module.agent import Agent

# Khởi tạo agent
agent = Agent()

# Prompt bóc tách địa chỉ
prompt_template = """
Bạn là một trợ lý phân tích dữ liệu cứu hộ. Hãy bóc tách địa chỉ sau thành định dạng JSON.
Địa chỉ: {location}

Yêu cầu trả về JSON với các trường:
- "tinh_tp": Tỉnh hoặc Thành phố trực thuộc trung ương.
- "quan_huyen": Quận, Huyện hoặc Thị xã.
- "phuong_xa": Phường, Xã hoặc Thị trấn.
- "nha": Số nhà, ngõ, tên đường hoặc các mô tả chi tiết vị trí nhỏ hơn xã.

Chỉ trả về duy nhất khối JSON, không giải thích gì thêm.
Kết quả JSON:
"""

class Response(BaseModel):
    tinh_tp: str
    quan_huyen: str
    phuong_xa: str
    nha: str


def data_clean(
    input_file="data.csv",
    output_file="data_encoded_local.csv"
):
    # =======================
    # 1. ĐỌC DỮ LIỆU
    # =======================
    df = pd.read_csv(input_file)
    count_before = len(df)

    # =======================
    # 2. LỌC TRÙNG CONTENT
    # =======================
    df = df.drop_duplicates(subset=['content'], keep='first').copy()
    count_after = len(df)

    # =======================
    # 3. MÃ HÓA CONTACT
    def contact_has_phone(val):
        if pd.isna(val) or val == '[]':
            return 0
        digits = re.sub(r'[^0-9]', '', str(val))
        return 1 if len(digits) >= 7 else 0

    if 'contact' in df.columns:
        df['contact'] = df['contact'].apply(contact_has_phone)
    else:
        df['contact'] = 0

    # =======================
    # 4. BÓC TÁCH ĐỊA CHỈ
    # =======================
    def parse_address_local(location: str) -> dict:
        if pd.isna(location) or str(location).strip() == "":
            return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}
        try:
            user_prompt = prompt_template.format(location=location)
            message = agent.generate_sync(
                user_prompt=user_prompt,
                response_schema=Response
            )
            if hasattr(message, 'dict'):
                return message.dict()
            return dict(message)
        except Exception as e:
            print(f"Lỗi dòng: {location} -> {e}")
            return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}

    print(f"Bắt đầu bóc tách {len(df)} địa chỉ...")

    results = [parse_address_local(loc) for loc in df['location'].tolist()]
    df_address = pd.json_normalize(results)

    # =======================
    # 5. GỘP & LƯU FILE
    # =======================
    df_final = pd.concat([df.reset_index(drop=True), df_address], axis=1)
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

    # =======================
    # 6. LOG
    # =======================
    print(f"Số dòng ban đầu: {count_before}")
    print(f"Số dòng sau khi lọc trùng: {count_after}")
    print(f"Đã xóa: {count_before - count_after} dòng trùng")
    print(f"Hoàn thành! File xuất ra: {output_file}")

    return df_final


In [5]:
import pandas as pd 
df_encoded_local = pd.read_csv("data_encoded_local.csv")
df_encoded_local.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data_encoded_local.csv'

In [ ]:
import pandas as pd
import re

# 1. Đọc dữ liệu
df = pd.read_csv('data_encoded_local.csv')

# 2. Hàm chuyển đổi text sang điểm số
def convert_disaster_to_score(text):
    if pd.isna(text) or str(text).strip() == '':
        return 0 # Không rõ/Trống
    
    text = str(text).lower()
    # Xử lý lỗi chính tả
    text = text.replace('sạc lỡ', 'sạt lở').replace('sạc lở', 'sạt lở')
    
    # Mức 5: Lũ lụt, Sạt lở, Sập nhà, Hỏa hoạn
    # Logic: Chỉ cần xuất hiện 1 trong các từ khóa này là đánh 5 ngay
    if re.search(r'lũ|ngập|nước dâng|sạt|trượt|đất|sập|đổ|tốc mái|hỏa|cháy|nổ', text):
        return 5
        
    # Mức 3: Thiên tai
    if re.search(r'thiên tai|bão', text):
        return 3
        
    # Mức 1: Sự cố hạ tầng (điện, mạng, nước)
    if re.search(r'điện|mạng|internet|wifi|nước', text):
        return 1
        
    return 0

# 3. Thay thế trực tiếp cột disaster_type
df['disaster_type'] = df['disaster_type'].apply(convert_disaster_to_score)

# 4. Lưu file kết quả
df.to_csv('data_encoded_scored.csv', index=False)

In [ ]:
import pandas as pd
import re
import json
from pydantic import BaseModel
from module.agent import Agent

# Khởi tạo agent
agent = Agent()

# Prompt bóc tách địa chỉ
prompt_template = """
Bạn là một trợ lý phân tích dữ liệu cứu hộ. Hãy bóc tách địa chỉ sau thành định dạng JSON.
Địa chỉ: {location}

Yêu cầu trả về JSON với các trường:
- "tinh_tp": Tỉnh hoặc Thành phố trực thuộc trung ương.
- "quan_huyen": Quận, Huyện hoặc Thị xã.
- "phuong_xa": Phường, Xã hoặc Thị trấn.
- "nha": Số nhà, ngõ, tên đường hoặc các mô tả chi tiết vị trí nhỏ hơn xã.

Chỉ trả về duy nhất khối JSON, không giải thích gì thêm.
Kết quả JSON:
"""

class Response(BaseModel):
    tinh_tp: str
    quan_huyen: str
    phuong_xa: str
    nha: str


def data_clean(
    df,
    output_file="data_encoded_local.csv"
):
    # =======================
    # 1. ĐỌC DỮ LIỆU
    # =====================
    count_before = len(df)

    # =======================
    # 2. LỌC TRÙNG CONTENT
    # =======================
    df = df.drop_duplicates(subset=['content'], keep='first').copy()
    count_after = len(df)

    # =======================
    # 3. MÃ HÓA CONTACT
    def contact_has_phone(val):
        if pd.isna(val) or val == '[]':
            return 0
        digits = re.sub(r'[^0-9]', '', str(val))
        return 1 if len(digits) >= 7 else 0

    if 'contact' in df.columns:
        df['contact'] = df['contact'].apply(contact_has_phone)
    else:
        df['contact'] = 0

    # =======================
    # 4. BÓC TÁCH ĐỊA CHỈ
    # =======================
    def parse_address_local(location: str) -> dict:
        if pd.isna(location) or str(location).strip() == "":
            return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}
        try:
            user_prompt = prompt_template.format(location=location)
            message = agent.generate_sync(
                user_prompt=user_prompt,
                response_schema=Response
            )
            if hasattr(message, 'dict'):
                return message.dict()
            return dict(message)
        except Exception as e:
            print(f"Lỗi dòng: {location} -> {e}")
            return {"tinh_tp": None, "quan_huyen": None, "phuong_xa": None, "nha": None}

    print(f"Bắt đầu bóc tách {len(df)} địa chỉ...")

    results = [parse_address_local(loc) for loc in df['location'].tolist()]
    df_address = pd.json_normalize(results)

    # =======================
    # 5. GỘP & LƯU FILE
    # =======================
    df_final = pd.concat([df.reset_index(drop=True), df_address], axis=1)
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

    # =======================
    # 6. LOG
    # =======================
    print(f"Số dòng ban đầu: {count_before}")
    print(f"Số dòng sau khi lọc trùng: {count_after}")
    print(f"Đã xóa: {count_before - count_after} dòng trùng")
    print(f"Hoàn thành! File xuất ra: {output_file}")

    return df_final
data_clean()